***

## Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
import geopandas as gpd
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_columns', None)

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'Zillow')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Zillow Data')
path_main = os.path.join(path_sp, 'Data')


# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'Zillow')
    path_config  = os.path.join(path_code, 'config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'Zillow')
    path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

## Cost_1

***

In [ ]:
# Assign geographies

sacog_state = ["Sacramento, CA", "Yuba City, CA"]

ca_peers = ["Los Angeles, CA", "San Francisco, CA", "Riverside, CA", 
            "San Diego, CA", "Oxnard, CA", "Santa Rosa, CA", 
            "Vallejo, CA", "El Centro, CA", "Napa, CA"]

other_peers = ["Austin, TX", "Charlotte, NC", "Cincinnati, OH",
               "Cleveland, OH", "Columbus, OH", "Detroit, MI",
               "Indianapolis, IN", "Kansas City, KS", "Miami, FL",
               "Orlando, FL", "Phoenix, AZ", "Pittsburg, PA",
               "Portland, OR", "Salt Lake City, UT", "San Antonio, TX",
               "St. Louis, MO", "Tampa, FL"]

# More classifiers
sacog = ["Yuba City", "Sacramento"]
mtc   = ["San Francisco", "Santa Rosa", "Vallejo", "Napa"]
scag  = ["Los Angeles", "Riverside", "Oxnard", "El Centro"]

In [ ]:
################################################### Cost_1a file ##########################################################################



# Load CSV file
df_sales = pd.read_csv(os.path.join(path_raw, "Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"))
df_sales = df_sales[df_sales['RegionType'] == 'msa']
df_sales = pd.melt(df_sales, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], var_name='date_', value_name='Price')

# Subset and rename cols
df_sales = df_sales[['RegionName', 'StateName', 'date_', 'Price']]
df_sales.columns = ['Region', 'State', 'date_', 'Price']
df_sales = df_sales[df_sales['Region'].isin(sacog_state + ca_peers + other_peers)]

# Removing trailing text from Region
df_sales['Region'] = df_sales['Region'].str.replace(r',.*', '', regex=True)

# Mutate function to get MPO col
df_sales['MPO'] = np.where(df_sales['Region'].isin(sacog), 'SACOG'
                    , np.where(df_sales['Region'].isin(mtc), 'MTC'
                    , np.where(df_sales['Region'].isin(scag), 'SCAG'
                    , np.where(df_sales['Region'] == 'San Diego', 'SANDAG', df_sales['Region']))))

# Calculate median prices for each MPO

sacog_mn = df_sales[df_sales['MPO'] == 'SACOG'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
sacog_mn['State'] = 'CA'
sacog_mn['Region'] = 'SACOG Median'
sacog_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

mtc_mn = df_sales[df_sales['MPO'] == 'MTC'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
mtc_mn['State'] = 'CA'
mtc_mn['Region'] = 'MTC Median'
mtc_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

scag_mn = df_sales[df_sales['MPO'] == 'SCAG'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
scag_mn['State'] = 'CA'
scag_mn['Region'] = 'SCAG Median'
scag_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

# Combine all median dfs, and clean cols
df_sales = pd.concat([df_sales, sacog_mn, mtc_mn, scag_mn])
df_sales = df_sales.sort_values(by=['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, False])
df_sales = df_sales[['State', 'MPO', 'Region', 'date_', 'Price']]
df_sales = df_sales.reset_index(drop = True)
df_sales['date_'] = df_sales['date_'].astype('str')

df_sales2 = df_sales.copy()

df_sales2['date_'] = pd.to_datetime(df_sales2['date_'])
df_sales2['Year'] = df_sales2['date_'].dt.year
df_sales2 = df_sales2.drop('date_', axis = 1)
df_sales2 = df_sales2.groupby(['State', 'MPO', 'Region', 'Year'], as_index = False)['Price'].mean()
df_sales2 = df_sales2.sort_values(by=['State', 'MPO', 'Region', 'Year'], ascending = [True, True, True, False])


# with pd.ExcelWriter(os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost', 'Cost_1 Sales Price', 'Cost_1 MSA Zillow.xlsx'), engine='xlsxwriter') as writer:
# # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
#     df_sales .to_excel(writer, index = False, sheet_name = 'Monthly')
#     df_sales2.to_excel(writer, index = False, sheet_name = 'Annual')

df_sales2.columns = [col.lower() for col in df_sales2.columns]
# df_sales2.to_csv(os.path.join(path_agol, 'Cost_1_MSA_Zillow_Annual.csv'), index = False)

display(df_sales2.head(6))

In [ ]:
path_plots = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost', 'Cost_1 Sales Price', 'plots')

df_plot = df_sales.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])

x = 'date_'
y = 'price'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot
                 , x = x
                 , y = y
                 , color = color
                 # , line_dash = line_dash
                 , markers = False
                 , labels = labels
                )

fig.update_layout(title = 'Median Home Sale Price by MPO')

# fig.write_html(
#     os.path.join(
#         path_plots
#         , ''.join(['Cost_1' + '_'
#                    , 'Median Home Sale Price by MPO_'
#                    , 'line_'
#                    , '.html'])
#     )
# )
    
fig.show()

In [ ]:
# ################################################### Cost_1b file ##########################################################################

# # Define SACOG counties
# sacog_counties = ["El Dorado County", "Placer County", "Sacramento County",
#                   "Sutter County", "Yolo County", "Yuba County"]

# # Read the ZIP code shapefile
# gp_zips = gpd.read_file(os.path.join(path_raw, "California Zip Codes SHP", "California_Zip_Codes.shp"))

# # Read the ZIP price data
# df_sales_zip = pd.read_csv(os.path.join(path_raw, "Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"))

# # # Filter zip_price for SACOG counties and California state
# df_sales_zip = df_sales_zip[(df_sales_zip['CountyName'].isin(sacog_counties)) & (df_sales_zip['State'] == 'CA')]

# # # Ensure ZIP_CODE column is integer for merging
# gp_zips['ZIP_CODE'] = gp_zips['ZIP_CODE'].astype(int)
# df_sales_zip['RegionName'] = df_sales_zip['RegionName'].astype(int)

# # Merge the data with zip_codes
# vals = gp_zips.merge(df_sales_zip, left_on='ZIP_CODE', right_on='RegionName', how='inner')

# # Sort by ZIP_CODE
# vals = vals.sort_values('ZIP_CODE')

# # # Select relevant columns
# date_columns = [col for col in df_sales_zip.columns if col.startswith('2')]
# cols_to_select = ['ZIP_CODE', 'PO_NAME', 'City', 'CountyName'] + date_columns
# vals = vals[cols_to_select]


# # Melt the DataFrame
# vals = pd.melt(vals, id_vars=['ZIP_CODE', 'PO_NAME', 'City', 'CountyName'], var_name='Date', value_name='Price')

# # Rename columns
# vals.columns = ['Zip', 'PO', 'City', 'County', 'Date', 'Price']

# display(vals.head(6))

# # Filter zip_codes for the ones present in vals
# gp_zips = gp_zips[gp_zips['ZIP_CODE'].astype(int).isin(vals['Zip'])]
# display(gp_zips.head(6))

***

## Cost 2

***

In [ ]:
################################################### Cost_2a file ##########################################################################

# Read in data
df_rents = pd.read_csv(os.path.join(path_raw, "Metro_zori_uc_sfrcondomfr_sm_month.csv")) #nolint
df_rents = df_rents[df_rents['RegionType'] == "msa"]

# Melt data down
df_rents = pd.melt(df_rents, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'])

# Keep important cols and rename
df_rents = df_rents[['RegionName', 'StateName', 'variable', 'value']]
df_rents.columns = ["Region", "State", "date_", "Price"]

# Subset data 
df_rents = df_rents[df_rents['Region'].isin(sacog_state + ca_peers + other_peers)]

# Mutate function to make the MPO col
df_rents['MPO'] = np.where(df_rents['Region'].str.contains('|'.join(sacog)), 'SACOG',
                        np.where(df_rents['Region'].str.contains('|'.join(mtc)), 'MTC',
                                 np.where(df_rents['Region'].str.contains('|'.join(scag)), 'SCAG',
                                          np.where(df_rents['Region'] == 'San Diego', 'SANDAG', df_rents['Region']))))

# This is similar to before. We want to get the median price and subset accordingly based on MPO.
sacog_mn = df_rents[df_rents['MPO'] == 'SACOG'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
sacog_mn['State'] = 'CA'
sacog_mn['Region'] = 'SACOG Median'
sacog_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

mtc_mn = df_rents[df_rents['MPO'] == 'MTC'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
mtc_mn['State'] = 'CA'
mtc_mn['Region'] = 'MTC Median'
mtc_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

scag_mn = df_rents[df_rents['MPO'] == 'SCAG'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
scag_mn['State'] = 'CA'
scag_mn['Region'] = 'SCAG Median'
scag_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

# Concatenate data
df_rents = pd.concat([df_rents, sacog_mn, mtc_mn, scag_mn], ignore_index=True)

# Sort data
df_rents = df_rents.sort_values(by=['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, False])
df_rents = df_rents[['State', 'MPO', 'Region', 'date_', 'Price']]
df_rents = df_rents.reset_index(drop = True)
df_rents['date_'] = df_rents['date_'].astype('str')


df_rents2 = df_rents.copy()

df_rents2['date_'] = pd.to_datetime(df_rents2['date_'])
df_rents2['Year'] = df_rents2['date_'].dt.year
df_rents2 = df_rents2.drop('date_', axis = 1)
df_rents2 = df_rents2.groupby(['State', 'MPO', 'Region', 'Year'], as_index = False)['Price'].mean()
df_rents2 = df_rents2.sort_values(by=['State', 'MPO', 'Region', 'Year'], ascending = [True, True, True, False])


# with pd.ExcelWriter(os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost', 'Cost_2 Rent Prices', 'Cost_2 MSA Zillow.xlsx'), engine='xlsxwriter') as writer:
# # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
#     df_rents .to_excel(writer, index = False, sheet_name = 'Monthly')
#     df_rents2.to_excel(writer, index = False, sheet_name = 'Annual')


df_rents2.columns = [col.lower() for col in df_rents2.columns]
# df_rents2.to_csv(os.path.join(path_agol, 'Cost_2_MSA_Zillow_Annual.csv'), index = False)

# Reorder cols
display(df_rents2.head(6))

In [ ]:
df_rents.Region.unique()

In [ ]:
path_plots = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost', 'Cost_2 Rent Prices', 'plots')

df_plot = df_rents.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])

x = 'date_'
y = 'price'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Rent Price by MPO')
# fig.write_html(os.path.join(path_plots, ''.join(['Cost_2' + '_', 'Median Home Rent Price by MPO_', 'line_', '.html'])))
fig.show()



df_plot = df_rents2.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])

x = 'year'
y = 'price'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Annual Average Home Rent Price by MPO')
# fig.write_html(os.path.join(path_plots, ''.join(['Cost_2' + '_', 'Annual Average Home Rent Price by MPO_', 'line_', '.html'])))
fig.show()

In [ ]:
# # ################################################### Cost_2b file ##########################################################################

# # Read files in
# gp_zips = gpd.read_file(os.path.join(path_raw, "California Zip Codes SHP", "California_Zip_Codes.shp"))
# df_rents_zip = pd.read_csv(os.path.join(path_raw, "Zip_zori_uc_sfrcondomfr_sm_month.csv"))

# # Filter the zip_rent df for SACOG counties in CA
# df_rents_zip = df_rents_zip[(df_rents_zip['CountyName'].isin(sacog_counties)) & (df_rents_zip['State'] == "CA")]

# # Convert zip code to integer in both the dfs
# gp_zips['ZIP_CODE'] = gp_zips['ZIP_CODE'].astype(int)
# df_rents_zip['RegionName'] = df_rents_zip['RegionName'].astype(int)

# # Merge the zip codes df with the rent df. This will end up being what we use for the final frame. 
# vals = gp_zips.merge(df_rents_zip, left_on='ZIP_CODE', right_on='RegionName')

# # Arrange by zip
# vals = vals.sort_values(by='ZIP_CODE')

# # Take all of the date columns for meting later. We then subset VALS to only include info we want. 
# date_columns = [col for col in df_rents_zip.columns if col.startswith('2')]
# vals = vals[['ZIP_CODE', 'PO_NAME', 'City', 'CountyName'] + date_columns]

# # Melt the dataframe
# vals = pd.melt(vals, id_vars=['ZIP_CODE', 'PO_NAME', 'City', 'CountyName'], var_name='date_', value_name='Rent')

# # Rename columns
# vals.columns = ['Zip', 'PO', 'City', 'County', 'date_', 'Price']

# display(vals.head(6))

# # Filter zip_codes for the ones present in vals
# gp_zips = gp_zips[gp_zips['ZIP_CODE'].astype(int).isin(vals['Zip'])]
# display(gp_zips.head(6))
